In [2]:
import os
import torch
import cv2
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
import shutil
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import subprocess
from torchvision import transforms
from torchvision.transforms.functional import to_tensor
import sys
from PIL import Image
import torch.optim as optim

In [3]:
class VideoDataset(Dataset):
    def __init__(self, data, temp_data_folder, NUM_FRAMES=10, transform_frame=None, label_map=None,transform_video=None, video_fps=25, resolution='1920:1080'):
        self.data = data  # It should be a list of tuples where data[0] is the path to the video and data[1] is the label

        self.transform_frame = transform_frame  # Transformations to be done on the individual frames.
                                                # Recommended to use when transforms is required at frames level with some randomness, eg: Random Crop
                                                # Note: If the Dataset is showing tensor issue, try adding `ToTensor()` in transform.

        self.transform_video = transform_video  # Transformations to be done on the whole video.
                                                # Recommended to use when transforms is required at video level with some randomness, eg: Random Horizontal Flip

        self.NUM_FRAMES = NUM_FRAMES  # Number of frames to be extracted from the video

        self.fps = video_fps  # The fps at which the video will be saved by ffmpeg
                              # Note: Reducing this might give a small performance increase, which might add up when running it multiple times. But this will also lead to
                              # loss of some data, as some frames will be dropped by ffmpeg
        self.label_map = label_map
        self.resolution = resolution  # resolution at which ffmpeg will save the frames (could be the same as the video or different).
                                      # Note: Reducing this might give a small performance increase, which might add up when running it multiple times.

        self.temp_data_folder = temp_data_folder  # A temporary folder where ffmpeg can store the frames of the video
                                                  # NOTE: this folder is recommended to be empty as frames get deleted from the folder after they are loaded as tensor!!


    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        video_path = self.data[idx][0]
        video_file_name = os.path.basename(video_path)  # Assigns the name of the video to the variable
        output_video_path = os.path.join(self.temp_data_folder, video_file_name)  # creating a temp path in the temp_folder for saaving the frames of the video
        os.makedirs(output_video_path, exist_ok=True)  # Creating a folder with the name of the video in the temp_data_folder to save the frames of the video

        # Command to convert the video to frames
        fallback_cmd = [
                    'ffmpeg',
                    '-i', video_path,
                    '-vf', f'fps={self.fps},scale={self.resolution},format=yuv420p',
                    '-q:v', '2',
                    os.path.join(output_video_path, 'img_%05d.jpg')
        ]
        subprocess.run(fallback_cmd, check=True, stderr=subprocess.PIPE)

        video_frames = []

        video_images = sorted(os.listdir(output_video_path))

        frame_positions = np.linspace(0, len(video_images)-1, self.NUM_FRAMES, dtype=int)

        # Selecting the NUM_FRAMES from the video
        for n in frame_positions:
            img = video_images[int(n)]
            img_path = os.path.join(output_video_path, img)
            with Image.open(img_path) as pil_img:
                if self.transform_frame:
                    video_frames.append(self.transform_frame(pil_img))  # Note: Try adding ToTensor() in transform_frame, if any tensor related error arrises.
                    
                else:
                    video_frames.append(to_tensor(pil_img))

        try:
            video_frames = torch.stack(video_frames)  # Note: Try adding ToTensor() in transform_frame, if any tensor related error arrises.
            # video_frames = torch.stack(video_frames)   # (T, C, H, W)
            # video_frames = video_frames.mean(dim=0)   
        except TypeError:
            print(f"TypeError: Tried to stack {type(video_frames[0])}. Add ToTensor() in transform_frame!")
            sys.exit(1)
            return None, None


        if self.transform_video:
            video_frames = self.transform_video(video_frames)
        shutil.rmtree(output_video_path)  # Clean up temp folder

        label = self.data[idx][1]
        if self.label_map is not None:
            label = self.label_map[label]
        return video_frames, torch.tensor(label, dtype=torch.long)

In [4]:
df = pd.read_csv("train.csv")
data_tuple = list(zip(df.path, df.label))

In [5]:
label_map = {
    "geste_0": 0,
    "geste_1" :1,
    "geste_2": 2
}

In [6]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])

In [ ]:
dataset = VideoDataset(
    data=data_tuple,
    temp_data_folder="temp/",
    NUM_FRAMES=10,
    # transform_frame=transform,
    label_map=label_map,
    video_fps=30,
    resolution="112:112"

)

batch_size= 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [8]:
val_df = pd.read_csv("val.csv")
val_data = list(zip(val_df.path, val_df.label))

val_dataset = VideoDataset(
    data=val_data,
    temp_data_folder='temp/',
    NUM_FRAMES=30,
    label_map=label_map,
    video_fps=30,
    resolution='380:480'
)

batch_size = 32
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

In [9]:
# for video_frames, labels in dataloader:
#     print(labels)

In [10]:
class FrameCNN(nn.Module):
    """
    Lightweight CNN that maps a single RGB frame → feature vector.
    Global Average Pooling removes any dependency on input resolution.
    """
    def __init__(self, feature_dim: int = 256):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # 3 channels (RGB)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                               # H/2, W/2
 
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                               # H/4, W/4
 
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                               # H/8, W/8
 
            # Block 4
            nn.Conv2d(128, feature_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_dim),
            nn.ReLU(inplace=True),
        )
        # Global Average Pooling → output is always (B, feature_dim)
        # regardless of spatial input size.
        self.gap = nn.AdaptiveAvgPool2d(1)
 
    def forward(self, x):
        # x: (B, C, H, W)
        x = self.features(x)   # (B, feature_dim, h, w)
        x = self.gap(x)        # (B, feature_dim, 1, 1)
        x = x.flatten(1)       # (B, feature_dim)
        return x
 
 
class VideoClassifier(nn.Module):
    """
    Processes each frame independently with a shared CNN,
    then aggregates frame features via mean-pooling and classifies.
 
    Input shape : (B, T, C, H, W)
    Output shape: (B, num_classes)
    """
    def __init__(self, num_classes: int = 14, feature_dim: int = 256):
        super().__init__()
        self.frame_cnn = FrameCNN(feature_dim=feature_dim)
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )
 
    def forward(self, x):
        # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape
 
        # Merge batch and time dims so we process all frames at once
        x = x.view(B * T, C, H, W)          # (B*T, C, H, W)
        feats = self.frame_cnn(x)            # (B*T, feature_dim)
        feats = feats.view(B, T, -1)         # (B, T, feature_dim)
 
        # Temporal aggregation: simple mean pooling across frames
        feats = feats.mean(dim=1)            # (B, feature_dim)
 
        return self.classifier(feats)        # (B, num_classes)


In [11]:
device = "cuda"
model = VideoClassifier(num_classes=3).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
train_accs = []
val_accs = []
for epoch in range(1, 50 + 1):
    # ── Train ─────────────────────────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for videos, labels in dataloader:
        # videos : (B, T, C, H, W)
        # labels : (B,)
        videos, labels = videos.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(videos)                    # (B, num_classes)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item() * videos.size(0)
        preds          = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total   += videos.size(0)

    scheduler.step()
    avg_train_loss = train_loss / train_total
    train_acc      = train_correct / train_total
    train_accs.append(train_acc)
    # validate
    model.eval()

    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for videos, labels in val_dataloader:
            videos, labels = videos.to(device), labels.to(device)
            outputs = model(videos)                    # (B, num_classes)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * videos.size(0)
            preds = outputs.argmax(dim=1)
            # _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

    avg_loss = val_loss / val_total

    accuracy = (val_correct / val_total)*100
    val_accs.append(accuracy)

    if epoch%10 == 0:
        print(f"Epoch : {epoch}")
        print(f"Average train loss: {avg_train_loss} \nTrain  Accuracy : {train_acc}") 
        print("----------------------------------------------------------------------")   
        print(f"Average loss: {avg_loss} \n Accuracy : {accuracy}")


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.88 GiB. GPU 0 has a total capacity of 7.65 GiB of which 170.38 MiB is free. Process 1886943 has 4.63 GiB memory in use. Including non-PyTorch memory, this process has 2.11 GiB memory in use. Of the allocated memory 1.96 GiB is allocated by PyTorch, and 20.38 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
torch.save(model.state_dict(), "model.pth")

In [ ]:
val_df = pd.read_csv("test.csv")
val_data = list(zip(val_df.path, val_df.label))

val_dataset = VideoDataset(
    data=val_data,
    temp_data_folder='temp/',
    NUM_FRAMES=10,
    label_map=label_map,
    video_fps=30,
    resolution='112:112'
)

batch_size = 32
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
model.eval()

val_loss, val_correct, val_total = 0.0, 0, 0

with torch.no_grad():
    for videos, labels in val_dataloader:
        videos, labels = videos.to(device), labels.to(device)
        outputs = model(videos)                    # (B, num_classes)
        loss = criterion(outputs, labels)
        
        val_loss += loss.item() * videos.size(0)
        preds = outputs.argmax(dim=1)
        # _, predicted = torch.max(outputs, 1)
        val_total += labels.size(0)
        val_correct += (preds == labels).sum().item()

avg_loss = val_loss / val_total

accuracy = (val_correct / val_total)*100

print(accuracy)

100.0
